In [1]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset
dataset = load_dataset("beomi/KoAlpaca-v1.1a")
dataset

c:\Users\playdata2\miniconda3\envs\openai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 21155
    })
})

In [3]:
print(dataset['train']['instruction'][1])
print(dataset['train']['output'][1])
print(dataset['train']['url'][1])

스웨터의 유래는 어디에서 시작되었나요?
스웨터의 유래는 14세기경 북유럽항구지역에서 어망을 짜던 기술을 의복에 활용하면서 시작되었습니다. 노동자들의 방한복에서 시작된 스웨터는 여가생활과 스포츠의 붐에 힘입어 대중화되었습니다. 이후, 겨울철 이너웨어의 대명사가 되었습니다. 스웨터는 짜서(Knit) 만든 옷을 말하며, 어부들의 방한복으로 짜여졌던 스웨터 중에서도 스코틀랜드 해안지방의 여인들은 바다로 나가는 남편이나 연인, 자식들에게 무사히 돌아올 것을 기원하며 로프나 닻 무늬를 정성껏 짜넣었다고 합니다. 그 실용성과 정성이 오늘에까지 이어지고 있습니다.
https://kin.naver.com/qna/detail.naver?d1id=11&dirId=11080102&docId=47833655


In [4]:
# openai 파인튜닝
# SFT(Supervised Fine-Tuning) 포멧
# json
{
    "messages":[
        {'role':'user', 'content':'질문'},
        {'role':'assistant', 'content':'정답'},
    ]
}

{'messages': [{'role': 'user', 'content': '질문'},
  {'role': 'assistant', 'content': '정답'}]}

In [5]:
for data in dataset['train']:
    print(data)
    break

{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?', 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268'}


In [6]:
import json
output_file = 'KoAlpaca.jsonl'
with open(output_file,'w',encoding='utf-8') as f:
    for data in dataset['train']:                
        data = {
            'messages':[
                {'role':'user','content':data['instruction']},
                {'role':'assistant','content':data['output']}
            ]
        }
        f.write(json.dumps(data, ensure_ascii=False) + '\n')

In [7]:
# openai에 업로드
from openai import OpenAI
client = OpenAI()
file = client.files.create(
    file = open('KoAlpaca.jsonl','rb'),
    purpose='fine-tune'
)
print(f'업로드된 파일 id : {file.id}')

업로드된 파일 id : file-FY5q41cDhywRPMYRNCzNNz


In [8]:
# 파인 튜닝
job = client.fine_tuning.jobs.create(
    training_file= file.id,
    model = 'gpt-4o-mini-2024-07-18'
)
print(f'파인튜닝 job id : {job.id}')

파인튜닝 job id : ftjob-CDEBlCu15WPpZfHzZkHGLowY
